#### Setup

First, let's import the necessary libraries and set up our connection to the OpenAI API:

In [13]:
from openai import OpenAI
import json

client = OpenAI()

def get_completion(prompt,
                   system_prompt="You are a helpful assistant.",
                   json_mode=False):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": prompt},
    ]

    if json_mode:
        response = client.chat.completions.create(
            model="gpt-6-luna",
            messages=messages,
            response_format={"type": "json_object"},
        )
    else:
        response = client.chat.completions.create(
            model="gpt-6-luna",
            messages=messages,
        )
    if response.choices[0].message.content is None:
        raise ValueError("The model returned no response.")
    return response.choices[0].message.content


#### User Input

Here's where we'll accept the user's parameters for their grocery list.

In [14]:

user_inputs = {
    "people": 5,
    "days": 7,
    "daily_calories": 2000,
    "allergens": ["gluten", "shellfish", "glucose"],
    "budget_usd": 200,
    "pantry": ["rice", "sugar", "soda"]
}

#### Step 1

Here we draft a meal plan for the week that should align with the user input's parameters.

In [15]:
schema_string = """
{
  "menu": [
    {
      "day": "",
      "dish": "",
      "ingredients": [],
      "calories": 0,
      "est_cost_usd": 0
    }
  ],
  "total_cost": 0
}
"""

def draft_plan(params):


    system_prompt = """You are a registered dietician who writes weekly dinner plans."""

    user_prompt = f"""
    <instructions>
 ## YOUR SOLUTION TO STEP 1.1 HERE ##
 Your task is to create a  7–day **dinner** plan for a family of {params["people"]} people.

 ## YOUR SOLUTION TO STEP 1.1 HERE ##
- Assign around {params["daily_calories"]} calories *per person* each day
 ## YOUR SOLUTION TO STEP 1.1 HERE ##
- Keep your estimated total cost less than or equal to ${params["budget_usd"]}  
- One or more of the family's members have the following allergies: {", ".join(params["allergens"])}.
Do not include any of these allergens in your plan.
IMPORTANT – we already have these ingredients on hand:  

## YOUR SOLUTION TO STEP 1.2 HERE ##
{", ".join(params["pantry"])}
Use as MANY of those pantry items as you reasonably can while still meeting the other constraints.
    </instructions>

    <schema>
Return your answer as pure JSON (no markdown, no comments) matching
exactly this schema:
## YOUR SOLUTION TO STEP 1.3 HERE ##
{schema_string}
    </schema>
"""

    raw_json = get_completion(user_prompt,
                              system_prompt,
                              json_mode=True
                              )

    if raw_json is None:
      raise ValueError("The model returned no response.")
    return json.loads(raw_json)


#### Step 2
Now, a revising step looks at the first draft and generates feedback. There are two types of feedback: fixes and suggestions. Both must be addressed, but suggestions are secondary to fixes, and need only be implemented when they don't conflict with the fixes.

In [16]:
def critique_plan(plan, params):

    system_prompt = """You are a stern dietary QA inspector."""

    user_prompt = f"""
    <instructions>
    Here is the proposed plan (raw JSON):

## YOUR SOLUTION TO STEP 2.5 HERE ##
{json.dumps(plan)}

Check for rule violations:

## YOUR SOLUTION TO STEP 2.6 HERE ##
1. The total_cost must be less than or equal to {params["budget_usd"]}
## YOUR SOLUTION TO STEP 2.6 HERE ##
2. Each day's calories must be within ±15 % of {params["daily_calories"]}
## YOUR SOLUTION TO STEP 2.6 HERE ##
3. NONE of these allergens may appear: {', '.join(params["allergens"])}
## YOUR SOLUTION TO STEP 2.6 HERE ##
4. *Every* pantry item ({', '.join(params["pantry"])}) must appear at
   least once in the week.  Bonus points for using them more often.

Place any violation inside **fixes**.

Then think of nice-to-have tweaks (better variety, seasonal veg, quicker
prep, etc.). Put those ideas in **suggestions**.
</instructions>

<schema>
Reply with JSON matching this schema:
{{"fixes":[], "suggestions":[]}}
Output ONLY the JSON. No markdown backticks, no comments, no extra text.
</schema>
"""

    raw = get_completion(user_prompt,
                         system_prompt,
                         json_mode=True
                         )

    if raw is None:
        raise ValueError("The model returned no response.")
    return json.loads(raw)



#### Step 3
Now the workflow will apply the revisions with the following prompt. We'll also include a function that will create a grocery list after the meal plan is created.

In [17]:

def revise_plan(plan, fixes, suggestions, params):
    system_prompt = """You are a senior meal planner applying corrections."""
    user_prompt = f"""
    <instructions>
Your task is to apply the following corrections to this meal plan:
## YOUR SOLUTION HERE ##
{json.dumps(plan)}

Here are the mandatory fixes:
## YOUR SOLUTION HERE ##
{json.dumps(fixes)}

Optional but welcome suggestions:
## YOUR SOLUTION HERE ##
{json.dumps(suggestions)}
Apply every fix.  Use suggestions only if they don't break the rules.
Return the **updated** plan as plain JSON (same schema, no extra text).
</instructions>

<schema>
Return your answer as pure JSON (no markdown, no comments) matching
exactly this schema:
## YOUR SOLUTION HERE ##
{schema_string}
</schema>
"""

    ## YOUR SOLUTION HERE ##
    raw = get_completion(user_prompt,
                         system_prompt,
                         json_mode=True)
    
    if raw is None:
        raise ValueError("The model returned no response.")
    return json.loads(raw)


def build_grocery_list(plan, pantry):

    system_prompt = """You are a helpful kitchen assistant."""

    user_prompt = f"""
    <instructions>
Using the dinner plan below, create a shopping list for one week.
Do **not** include anything already in the pantry:
({', '.join(pantry)}).

Return JSON exactly like:
{{"shopping_list":[{{"item":"","estimated_qty":""}}] }}

Return ONLY a valid JSON object.

Do not include:
- Markdown
- Triple backticks
- Explanations
- Introductory text

Return exactly one JSON object.

Plan JSON:
{json.dumps(plan)}
</instructions>
"""

    raw = get_completion(user_prompt,
                         system_prompt,
                         json_mode=True)
    if raw is None:
        raise ValueError("The model returned no response.")
    return json.loads(raw)


#### Step 4
Finally, let's assemble the workflow.

In [18]:

MAX_PASSES = 3

def build_plan(params):

    ## YOUR SOLUTION HERE ##
    plan = draft_plan(params)

    for _ in range(MAX_PASSES):

        ## YOUR SOLUTION HERE ##
        critique = critique_plan(plan, params)

        if not critique["fixes"] and not critique["suggestions"]:
            break

        ## YOUR SOLUTION HERE ##
        plan = revise_plan(plan, critique["fixes"], critique["suggestions"], params)

        return plan


if __name__ == "__main__":

    final_plan = build_plan(user_inputs)

    print("\nFINAL PLAN")
    print(json.dumps(final_plan, indent=2))

    groceries = build_grocery_list(final_plan, user_inputs["pantry"])

    print("\nGROCERY LIST")
    print(json.dumps(groceries, indent=2))



FINAL PLAN
{
  "menu": [
    {
      "day": "Monday",
      "dish": "Roasted chicken thighs with rice and vegetables",
      "ingredients": [
        "8.75 cups dry rice (on hand)",
        "2.5 lb plain chicken thighs",
        "10 tbsp olive oil",
        "2 lb mixed non-starchy vegetables",
        "Salt, pepper, garlic, and dried herbs"
      ],
      "calories": 1970,
      "est_cost_usd": 23
    },
    {
      "day": "Tuesday",
      "dish": "Beef, black bean, and rice bowls with avocado",
      "ingredients": [
        "6.25 cups dry rice (on hand)",
        "2.5 lb plain ground beef",
        "5 cups cooked black beans, prepared from dried beans",
        "3 avocados",
        "5 tbsp olive oil",
        "Onions, bell peppers, salt, pepper, and chili spices"
      ],
      "calories": 1935,
      "est_cost_usd": 29
    },
    {
      "day": "Wednesday",
      "dish": "Pork, potato, and rice plates with vegetables",
      "ingredients": [
        "7.5 cups dry rice (on hand)",
